# gpt-oss-20b 模型介绍

模型名称：`openai/gpt-oss-20b`

这个模型是本仓库的第二条主线。它和 Qwen2.5-7B 最大的区别是：它是 MoE 模型，使用 Harmony 格式，并且涉及 MXFP4 和 reasoning effort。

## 1. 为什么单独成套

gpt-oss-20b 不能简单照搬 Qwen2.5-7B 的写法。

- 它不是普通 dense 模型，而是 MoE 模型。
- 它使用 Harmony 对话格式，不能随便套 ChatML。
- 它支持 reasoning effort，推理时要理解 low、medium、high 的差异。
- 它涉及 MXFP4，这和 QLoRA 里的 bitsandbytes 4bit 不是一回事。
- 它在 Ollama 和 vLLM 中有专门支持路径。

## 2. MoE 内部结构

MoE 是 Mixture-of-Experts。可以把它理解为模型内部有多个 expert，每次处理 token 时，由 router 选择部分 expert 参与计算。

关键理解：

- 总参数量约 20B 级别，但不是每个 token 都激活全部参数。
- active parameters 约 3.6B 级别，更接近单次 token 计算成本。
- router 决定 token 进入哪些 expert。
- 总参数影响模型文件体积和加载成本，active parameters 影响单步计算成本。

## 3. Harmony 格式

gpt-oss 使用 Harmony 格式组织对话。模板错误时，模型可能仍然能生成文本，但角色边界、推理内容、最终回答格式可能不稳定。

因此 gpt-oss 的训练和推理 notebook 必须单独说明：

- system、developer、user、assistant 的角色关系。
- reasoning effort 放在哪里。
- 输出中哪些部分是推理过程，哪些部分是最终回答。
- 为什么不能直接把普通 ChatML 当成 Harmony。

## 4. MXFP4

MXFP4 是 gpt-oss 相关的重要权重压缩格式。它不是 QLoRA 的 bitsandbytes 4bit。

新人需要记住：

- QLoRA 的 4bit 主要是训练时低显存加载方案。
- MXFP4 是 gpt-oss 权重和推理生态中的重点格式。
- 老 GPU 或 fp16 环境可能需要额外精度处理。
- 如果只是本地运行 gpt-oss，Ollama 的 `gpt-oss:20b` 是优先路线之一。

## 5. reasoning effort

gpt-oss 支持 reasoning effort，用来控制推理深度和响应速度。

- `low`：速度优先，适合简单问答。
- `medium`：平衡速度和质量，适合作为默认选择。
- `high`：推理更充分，但更慢、生成 token 更多。

后续部署 notebook 必须展示这三个设置如何影响输出。